# Charged particles in E and B fields

*em · beginner* · [open in Tensorizer](https://jackyko1991.github.io/tensorizer/#/em/lorentz_force)

The Lorentz force on a batch of charged particles in crossed electric and magnetic fields. The cross product is written with the Levi-Civita tensor $\varepsilon_{ijk}$, so one einsum gives the magnetic force on every particle: each one spirals in a cyclotron helix while the whole swarm drifts along $\mathbf E\times\mathbf B$.

**Vector form**

$$m\,\dot{\mathbf v} = q\,(\mathbf E + \mathbf v \times \mathbf B)$$

**Index (tensor) form**

$$m_b\,\dot v_{bi} = q_b\,\big(E_i + \varepsilon_{ijk}\,v_{bj}\,B_k\big)$$

| Symbol | Code | Shape | Meaning |
|---|---|---|---|
| $x_{bi}$ | `y[:, 0, :]` | (P, 3) | position of particle $b$, component $i \in \{x, y, z\}$ |
| $v_{bi}$ | `y[:, 1, :]` | (P, 3) | velocity of particle $b$ |
| $q_b,\ m_b$ | `q, m` | (P,) | charge and mass of each particle |
| $E_i,\ B_k$ | `E, B` | (3,) | uniform fields: $\mathbf E$ along $y$, $\mathbf B$ along $z$ |
| $\varepsilon_{ijk}$ | `levi_civita(dev)` | (3, 3, 3) | $+1$ for even permutations of $(x, y, z)$, $-1$ for odd ones, 0 if an index repeats |
| $\varepsilon_{ijk}v_{bj}B_k$ | `einsum('ijk,bj,k->bi', eps, v, B)` | (P, 3) | repeated $j$ and $k$ ⇒ summed: the cross product $\mathbf v\times\mathbf B$ |
| $\varepsilon_{ijk}E_jB_k/B_lB_l$ | `drift_velocity(E, B, eps)` | (3,) | $\mathbf E\times\mathbf B$ drift velocity, the same for every charge |

Units are dimensionless ($q$, $m$, $E$, $B$ of order one). With $\mathbf E = 0$ each particle circles at the cyclotron frequency $\omega = \lvert q\rvert B/m$ with radius $mv_\perp/(\lvert q\rvert B)$; positive and negative charges turn in opposite senses. Adding $\mathbf E\perp\mathbf B$ makes every guiding centre drift with velocity $\mathbf E\times\mathbf B/B^2$, independent of charge and mass.


## Setup

On Colab, choose **Runtime → Change runtime type → GPU** to run on a GPU — the code uses it automatically.

In [ ]:
import os
import subprocess
import sys

# Shared helpers (problems/_common) come from the repository; skip the clone if we are already inside it.
if not os.path.isdir("problems"):
    if not os.path.isdir("tensorizer"):
        subprocess.run(["git", "clone", "--depth", "1", "https://github.com/jackyko1991/tensorizer.git"], check=True)
    os.chdir("tensorizer")
sys.path.insert(0, os.getcwd())
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "pyyaml", "tqdm"], check=True)

import torch

print("torch", torch.__version__, "· device:", "cuda" if torch.cuda.is_available() else "cpu")


## 1. `main.py` — main loop

In [ ]:
%%writefile problems/em/lorentz_force/main.py
"""Main loop: parameters → tensors → solve → plot."""

from pathlib import Path

from problems._common.runtime import cli, device

from . import plot, solve, tensors
from .equation import cyclotron_frequency, drift_velocity


def run(params: dict, out_dir: Path) -> Path:
    dev = device()
    eps = tensors.levi_civita(dev)
    E, B = tensors.fields(params["E"], params["B"], dev)
    q, m = tensors.particles(params["n_particles"], dev)
    y0 = tensors.initial_state(params["n_particles"], params["v_par"], dev)

    times, traj = solve.solve(y0, q, m, E, B, eps, params["t_max"])

    drift = drift_velocity(E, B, eps)
    for qb, mb, w in zip(q.tolist(), m.tolist(), cyclotron_frequency(q, m, B).tolist()):
        print(f"q {qb:+.0f}  m {mb:3.1f}  cyclotron omega = {w:.3f}")
    print(f"ExB drift velocity: {[round(u, 3) for u in drift.tolist()]}")
    return plot.animate(times, traj, q, m, drift, out_dir / "lorentz_force.gif")


if __name__ == "__main__":
    cli(run, __file__)


## 2. `equation.py` — physics → tensor translation

In [ ]:
%%writefile problems/em/lorentz_force/equation.py
r"""Physics → tensor translation.

The Lorentz force on a particle of charge q and mass m,

    m \dot{\mathbf v} = q (\mathbf E + \mathbf v \times \mathbf B),

written in index notation for a batch of particles b and spatial components i, j, k:

    m_b \dot v_{bi} = q_b ( E_i + \varepsilon_{ijk} v_{bj} B_k )

The cross product is the Levi-Civita tensor contracted with two vectors: j and k are
repeated, so both are summed — exactly einsum('ijk,bj,k->bi'). As a first-order system
with state y = (x, v):

    \dot x_{bi} = v_{bi},    \dot v_{bi} = (q_b / m_b)(E_i + \varepsilon_{ijk} v_{bj} B_k)
"""

import torch


def cross(eps: torch.Tensor, v: torch.Tensor, B: torch.Tensor) -> torch.Tensor:
    r"""(v × B)_{bi} = \varepsilon_{ijk} v_{bj} B_k.   eps: (3, 3, 3), v: (P, 3), B: (3,) → (P, 3)"""
    return torch.einsum("ijk,bj,k->bi", eps, v, B)


def acceleration(v, q, m, E, B, eps) -> torch.Tensor:
    r"""\dot v_{bi} = (q_b/m_b)(E_i + \varepsilon_{ijk} v_{bj} B_k).   v: (P, 3), q, m: (P,) → (P, 3)"""
    force_per_charge = E + cross(eps, v, B)  # (P, 3)  E_i broadcast over b
    return torch.einsum("b,bi->bi", q / m, force_per_charge)


def rhs(y: torch.Tensor, q, m, E, B, eps) -> torch.Tensor:
    """dy/dt for the state y of shape (P, 2, 3): y[:, 0] = position, y[:, 1] = velocity."""
    x_dot = y[:, 1]
    v_dot = acceleration(y[:, 1], q, m, E, B, eps)
    return torch.stack([x_dot, v_dot], dim=1)


def cyclotron_frequency(q: torch.Tensor, m: torch.Tensor, B: torch.Tensor) -> torch.Tensor:
    r"""ω_b = |q_b| \sqrt{B_k B_k} / m_b.   → (P,)"""
    return q.abs() * torch.sqrt(torch.einsum("k,k->", B, B)) / m


def drift_velocity(E: torch.Tensor, B: torch.Tensor, eps: torch.Tensor) -> torch.Tensor:
    r"""u_i = \varepsilon_{ijk} E_j B_k / (B_l B_l): the E×B drift of every guiding centre.   → (3,)"""
    return torch.einsum("ijk,j,k->i", eps, E, B) / torch.einsum("l,l->", B, B)


## 3. `tensors.py` — PyTorch tensors

In [ ]:
%%writefile problems/em/lorentz_force/tensors.py
"""Build the concrete tensors: Levi-Civita symbol, uniform fields and the particle batch."""

import itertools

import torch

# (charge, mass) for up to six particles: opposite signs, double charge, heavier and lighter ones
SPECIES = [(1.0, 1.0), (-1.0, 1.0), (2.0, 1.0), (1.0, 2.0), (-2.0, 1.0), (1.0, 0.5)]
EVEN = [(0, 1, 2), (1, 2, 0), (2, 0, 1)]  # cyclic permutations of (x, y, z)


def levi_civita(device: torch.device) -> torch.Tensor:
    """ε_ijk of shape (3, 3, 3): +1 for even permutations, −1 for odd ones, 0 when an index repeats."""
    eps = torch.zeros(3, 3, 3, device=device)
    for i, j, k in itertools.permutations(range(3)):
        eps[i, j, k] = 1.0 if (i, j, k) in EVEN else -1.0
    return eps


def fields(E: float, B: float, device: torch.device) -> tuple[torch.Tensor, torch.Tensor]:
    """E_i = (0, E, 0) and B_k = (0, 0, B): crossed fields, each of shape (3,)."""
    return torch.tensor([0.0, E, 0.0], device=device), torch.tensor([0.0, 0.0, B], device=device)


def particles(n: int, device: torch.device) -> tuple[torch.Tensor, torch.Tensor]:
    """Charges q_b and masses m_b of the first n species, each of shape (P,)."""
    q, m = zip(*SPECIES[:n])
    return torch.tensor(q, device=device), torch.tensor(m, device=device)


def initial_state(n: int, v_par: float, device: torch.device) -> torch.Tensor:
    """y0 of shape (P, 2, 3): every particle starts at the origin.

    y0[b, 0, :] = (0, 0, 0)          position x_{bi}
    y0[b, 1, :] = (0, 1, v_par)      velocity v_{bi}: unit speed ⊥ B plus a component along B
    """
    position = torch.zeros(n, 3, device=device)
    velocity = torch.tensor([0.0, 1.0, v_par], device=device).repeat(n, 1)  # (P, 3)
    return torch.stack([position, velocity], dim=1)


## 4. `solve.py` — solver

In [ ]:
%%writefile problems/em/lorentz_force/solve.py
"""Integrate the batched Lorentz-force ODE with 4th-order Runge–Kutta."""

import torch

from problems._common.integrators import integrate

from .equation import rhs


def solve(y0, q, m, E, B, eps, t_max: float, dt: float = 0.01) -> tuple[torch.Tensor, torch.Tensor]:
    """Return (times (T,), trajectory (T, P, 2, 3)).

    All P particles advance together: one RK4 step updates the whole (P, 2, 3) tensor.
    """
    n_steps = int(round(t_max / dt))
    traj = integrate(lambda y, t: rhs(y, q, m, E, B, eps), y0, dt, n_steps)  # (T, P, 2, 3)
    times = torch.arange(n_steps + 1, device=y0.device) * dt
    return times, traj


## 5. `plot.py` — 2D/3D plot → GIF

In [ ]:
%%writefile problems/em/lorentz_force/plot.py
"""Animate the helices in 3D: each particle leaves a trail, labelled by its charge and mass."""

from pathlib import Path

import matplotlib.pyplot as plt
import torch

from problems._common.plotting import frame_indices, save_gif


def animate(times, traj, q, m, drift: torch.Tensor, path: Path) -> Path:
    pos = traj[:, :, 0, :].cpu().numpy()  # (T, P, 3)
    t = times.cpu().numpy()

    fig = plt.figure(figsize=(7, 4))
    ax = fig.add_subplot(projection="3d")
    lo, hi = pos.min(axis=(0, 1)), pos.max(axis=(0, 1))
    pad = 0.05 * (hi - lo).max() + 0.1
    ax.set_xlim(lo[0] - pad, hi[0] + pad)
    ax.set_ylim(lo[1] - pad, hi[1] + pad)
    ax.set_zlim(lo[2] - pad, hi[2] + pad)
    ax.set_xlabel("x  (E×B)")
    ax.set_ylabel("y  (E)")
    ax.set_zlabel("z  (B)")
    ax.view_init(elev=22, azim=-60)

    labels = [f"q={qb:+.0f}, m={mb:g}" for qb, mb in zip(q.tolist(), m.tolist())]
    lines = [ax.plot([], [], [], lw=1.4, label=lab)[0] for lab in labels]
    dots = [ax.plot([], [], [], "o", ms=5, color=line.get_color())[0] for line in lines]
    ax.legend(loc="upper left", fontsize=7)
    u = drift.tolist()
    ax.set_title(f"E×B drift  u = ({u[0]:.2f}, {u[1]:.2f}, {u[2]:.2f})")
    clock = ax.text2D(0.85, 0.95, "", transform=ax.transAxes, va="top")

    def update(step: int):
        for b, (line, dot) in enumerate(zip(lines, dots)):
            line.set_data_3d(pos[: step + 1, b, 0], pos[: step + 1, b, 1], pos[: step + 1, b, 2])
            dot.set_data_3d([pos[step, b, 0]], [pos[step, b, 1]], [pos[step, b, 2]])
        clock.set_text(f"t = {t[step]:.1f}")

    fig.subplots_adjust(left=0.0, right=1.0, bottom=0.02, top=0.93)
    return save_gif(fig, update, frame_indices(len(t), 90), path)  # 3D frames are slow: 90 is plenty


## Run

Change the parameters (or any file above) and run again.

In [ ]:
import importlib
from pathlib import Path

from IPython.display import Image, display

from problems._common.runtime import load_meta, resolve_params

params = {
    'B': 1.0,
    'E': 0.3,
    'v_par': 0.15,
    'n_particles': 4,
    't_max': 25.0
}

# Re-import so edits in the %%writefile cells above take effect.
for name in [m for m in sys.modules if m.startswith("problems.em.lorentz_force")]:
    del sys.modules[name]
main = importlib.import_module("problems.em.lorentz_force.main")
problem_dir = Path("problems/em/lorentz_force")
output = main.run(resolve_params(load_meta(problem_dir), params), Path("out"))
display(Image(filename=str(output)))
